# pybtex

In [ ]:
# Example 1: the `pybtex` command turns a LaTeX .aux file into a .bbl with a Python style (-l python), no TeX needed.
import pathlib, shutil, subprocess, tempfile

import pybtex

BIB = r"""@book{knuth1984, author = {Knuth, Donald E.}, title = {The {\TeX}book}, publisher = {Addison-Wesley}, year = {1984}}
@article{godel1931, author = {G{\"o}del, Kurt}, title = {{\"U}ber formal unentscheidbare S{\"a}tze},
  journal = {Monatshefte f{\"u}r Mathematik und Physik}, volume = {38}, pages = {173--198}, year = {1931}}
@article{einstein1905, author = {Einstein, Albert and von Neumann, John}, title = {Zur Elektrodynamik bewegter K{\"o}rper},
  journal = {Annalen der Physik}, volume = {17}, year = {1905}}
"""
work = pathlib.Path(tempfile.mkdtemp(prefix="pybtex-nb-"))
(work / "refs.bib").write_text(BIB)
(work / "paper.aux").write_text("\\citation{godel1931}\n\\citation{knuth1984}\n\\bibstyle{plain}\n\\bibdata{refs}\n")
version = subprocess.run([shutil.which("pybtex"), "--version"], capture_output=True, text=True, timeout=120)
assert version.stdout.strip() == f"pybtex-{pybtex.__version__}", version
run = subprocess.run(["pybtex", "-l", "python", "paper.aux"], cwd=work, capture_output=True, text=True, timeout=120)
assert run.returncode == 0, run.stderr
bbl = (work / "paper.bbl").read_text()
print(bbl)
assert "\\bibitem[1]{godel1931}" in bbl and "\\bibitem[2]{knuth1984}" in bbl
assert "{Ü}ber formal unentscheidbare" in bbl  # the LaTeX accent was decoded (python/latexcodec)

In [ ]:
# Example 2: parsing BibTeX: entries, typed fields, and names split into first / von / last parts.
from pybtex.database import parse_string
from pybtex.richtext import Text

data = parse_string(BIB, "bibtex")
print(list(data.entries))
assert list(data.entries) == ["knuth1984", "godel1931", "einstein1905"]
authors = data.entries["einstein1905"].persons["author"]
print([(str(p), p.prelast_names, p.last_names) for p in authors])
assert [str(p) for p in authors] == ["Einstein, Albert", "von Neumann, John"]
assert authors[1].prelast_names == ["von"] and authors[1].last_names == ["Neumann"]
title = data.entries["godel1931"].fields["title"]
print(title, "->", Text.from_latex(title).render_as("text"))
assert Text.from_latex(title).render_as("text") == "Über formal unentscheidbare Sätze"

In [ ]:
# Example 3: the Python styles order and label entries, and each backend renders the same bibliography.
import io

from pybtex.plugin import find_plugin

def labels(style_name):
    style = find_plugin("pybtex.style.formatting", style_name)()
    return [(entry.key, entry.label) for entry in style.format_bibliography(data).entries]

print(labels("plain"), labels("unsrt"), labels("alpha"), sep="\n")
assert labels("plain") == [("einstein1905", "1"), ("godel1931", "2"), ("knuth1984", "3")]  # sorted by author
assert [key for key, _ in labels("unsrt")] == ["knuth1984", "godel1931", "einstein1905"]  # database order
assert dict(labels("alpha"))["knuth1984"] == "Knu84" and dict(labels("alpha"))["einstein1905"] == "EvN05"

formatted = find_plugin("pybtex.style.formatting", "plain")().format_bibliography(data)
for backend_name, expected in [("text", "[2] Kurt Gödel."), ("html", "<em>Annalen der Physik</em>"), ("markdown", "*Annalen der Physik*")]:
    stream = io.StringIO()
    find_plugin("pybtex.backends", backend_name)().write_to_stream(formatted, stream)
    assert expected in stream.getvalue(), (backend_name, stream.getvalue()[:300])
print(stream.getvalue())

In [ ]:
# Example 4: the database converts between BibTeX, YAML (via pyyaml) and BibTeXML without losing anything.
for fmt in ("bibtex", "yaml", "bibtexml"):
    text = data.to_string(fmt)
    assert parse_string(text, fmt) == data, fmt
    print(fmt, len(text), "characters, round trip equal")
print(data.to_string("yaml")[:200])

(work / "refs.yaml").unlink(missing_ok=True)
convert = subprocess.run(["pybtex-convert", "refs.bib", "refs.yaml"], cwd=work, capture_output=True, text=True, timeout=120)
assert convert.returncode == 0, convert.stderr
from pybtex.database import parse_file
assert parse_file(str(work / "refs.yaml"), "yaml") == data

In [ ]:
import importlib.metadata

from pybtex.plugin import PluginNotFound

backends = sorted(ep.name for ep in importlib.metadata.entry_points(group="pybtex.backends"))
print("backends:", backends)
assert {"html", "latex", "markdown", "plaintext"} <= set(backends)
try:
    find_plugin("pybtex.style.formatting", "abbrv")
except PluginNotFound as error:
    print("abbrv:", error)
else:
    raise AssertionError("pybtex ships no Python `abbrv` style; finding one means the plug-in lookup changed")

# Without -l python, `pybtex` reads the classic plain.bst, which is looked up with TeX's kpsewhich: no TeX here.
classic = subprocess.run(["pybtex", "paper.aux"], cwd=work, capture_output=True, text=True, timeout=120)
print("exit", classic.returncode, "-", classic.stderr.strip().removeprefix("ERROR: "))  # pybtex says "unable to open plain.bst"
assert classic.returncode == 1 and "unable to open plain.bst" in classic.stderr, classic